In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Setup Chrome
options = Options()
options.add_argument("--headless")  # Remove this if you want to see browser
driver = webdriver.Chrome(options=options)

# Output list
all_data = []

# Iterate over all 34 pages (0-indexed)
for page in range(0, 34):
    print(f"Scraping page {page+1}...")

    url = f"https://www.eib.org/en/projects/all/index?q=&sortColumn=statusDate&sortDir=desc&pageNumber={page}&itemPerPage=500&pageable=true&la=EN"
    driver.get(url)

    # Wait for data to load
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CLASS_NAME, "eib-project-list-item"))
        )
    except:
        print("⚠️ Timeout waiting for project data.")
        continue

    rows = driver.find_elements(By.CLASS_NAME, "eib-project-list-item")
    
    for row in rows:
        try:
            title = row.find_element(By.TAG_NAME, "a").text.strip()
            spans = row.find_elements(By.TAG_NAME, "span")
            divs = row.find_elements(By.TAG_NAME, "div")

            country = spans[1].text.strip() if len(spans) > 1 else ""
            sector = spans[2].text.strip() if len(spans) > 2 else ""
            amount = divs[1].text.strip() if len(divs) > 1 else ""
            status = divs[2].text.strip() if len(divs) > 2 else ""
            last_change = divs[3].text.strip() if len(divs) > 3 else ""

            all_data.append({
                "Title": title,
                "Country": country,
                "Sector": sector,
                "Signed Amount": amount,
                "Current Status": status,
                "Last Status Change": last_change
            })
        except Exception as e:
            print(f"⚠️ Skipped a row on page {page+1}: {e}")
            continue

# Quit driver
driver.quit()

# Convert to DataFrame
df = pd.DataFrame(all_data)

# Optional cleanup
def clean_amount(euro_str):
    try:
        return euro_str.replace("€", "").replace(",", "").replace("\xa0", "").strip()
    except:
        return euro_str

df["Signed Amount"] = df["Signed Amount"].apply(clean_amount)

# Save
df.to_excel("eib_projects_16767.xlsx", index=False)
print(f"✅ Successfully scraped {len(df)} rows and saved to 'eib_projects_16767.xlsx'")


Scraping page 1...
⚠️ Timeout waiting for project data.
Scraping page 2...
⚠️ Timeout waiting for project data.
Scraping page 3...
⚠️ Timeout waiting for project data.
Scraping page 4...
⚠️ Timeout waiting for project data.
Scraping page 5...
⚠️ Timeout waiting for project data.
Scraping page 6...
⚠️ Timeout waiting for project data.
Scraping page 7...
⚠️ Timeout waiting for project data.
Scraping page 8...
⚠️ Timeout waiting for project data.
Scraping page 9...
⚠️ Timeout waiting for project data.
Scraping page 10...
⚠️ Timeout waiting for project data.
Scraping page 11...
⚠️ Timeout waiting for project data.
Scraping page 12...
⚠️ Timeout waiting for project data.
Scraping page 13...
⚠️ Timeout waiting for project data.
Scraping page 14...
⚠️ Timeout waiting for project data.
Scraping page 15...
⚠️ Timeout waiting for project data.
Scraping page 16...
⚠️ Timeout waiting for project data.
Scraping page 17...
⚠️ Timeout waiting for project data.
Scraping page 18...
⚠️ Timeout waiting f

KeyError: 'Signed Amount'

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time

# Configuration
BASE_URL = "https://www.eib.org/en/projects/all/index"
PARAMS = {
    'q': '',
    'sortColumn': 'statusDate',
    'sortDir': 'desc',
    'itemPerPage': 25,
    'pageable': 'true',
    'la': 'EN',
    'deLa': 'EN',
    # Other parameters remain constant
}
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

def scrape_eib_projects():
    with open('eib_projects.csv', 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Project Name', 'Country', 'Sector', 'Signed Amount', 'Status', 'Status Date']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for page in range(0, 671):  # Pages 0 to 670
            print(f"Scraping page {page + 1}/671...")
            PARAMS['pageNumber'] = page
            
            try:
                response = requests.get(BASE_URL, params=PARAMS, headers=HEADERS, timeout=30)
                response.raise_for_status()  # Check for HTTP errors
                
                soup = BeautifulSoup(response.text, 'html.parser')
                project_rows = soup.select('div.row-list.row-list-standard.row-contained')
                
                if not project_rows:
                    print(f"No projects found on page {page}. Stopping.")
                    break
                
                for row in project_rows:
                    # Skip header rows (no <a> tag in project name)
                    if not row.select_one('.col-md-4 a'):
                        continue
                    
                    project = {
                        'Project Name': row.select_one('.col-md-4 a').text.strip(),
                        'Country': row.select_one('.col-md-2:nth-of-type(1) .row-tags span').text.strip(),
                        'Sector': row.select_one('.col-md-2:nth-of-type(2) .row-tags span').text.strip(),
                        'Signed Amount': row.select_one('.col-md-2:nth-of-type(3) div').text.strip(),
                        'Status': row.select_one('.col-md-1:nth-of-type(1) div').text.strip(),
                        'Status Date': row.select_one('.col-md-1:nth-of-type(2) div').text.strip()
                    }
                    writer.writerow(project)
                
                time.sleep(1)  # Polite delay between requests
            
            except requests.exceptions.RequestException as e:
                print(f"Error on page {page}: {e}")
                continue

if __name__ == "__main__":
    scrape_eib_projects()


Scraping page 1/671...
No projects found on page 0. Stopping.


In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# --- CONFIGURATION ---
BASE_URL = "https://www.eib.org/en/projects/all/index"
TOTAL_PAGES = 671  # 0-based: 0 to 670
ITEMS_PER_PAGE = 25

chrome_options = Options()
chrome_options.add_argument("--headless")  # Run headless for speed
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

all_projects = []

for page_num in range(TOTAL_PAGES):
    print(f"Scraping page {page_num+1} of {TOTAL_PAGES}...")
    url = (
        f"{BASE_URL}?q=&sortColumn=null&sortDir=asc&pageNumber={page_num}"
        f"&itemPerPage={ITEMS_PER_PAGE}&pageable=true&la=EN&deLa=EN"
        "&yearFrom=&orYearFrom=true&yearTo=&orYearTo=true&orStatus=true"
        "&orRegions=true&orCountries=true&orSectors=true"
    )
    driver.get(url)
    time.sleep(2)  # Wait for JS to load

    # Find all project rows
    rows = driver.find_elements(By.CSS_SELECTOR, "div.row-list.row-list-standard.row-contained")
    for row in rows:
        # Skip header rows (no <a> tag)
        try:
            name = row.find_element(By.CSS_SELECTOR, ".col-md-4 a").text.strip()
        except:
            continue

        cols_2 = row.find_elements(By.CSS_SELECTOR, ".col-md-2")
        country = cols_2[0].text.strip() if len(cols_2) > 0 else ""
        sector = cols_2[1].text.strip() if len(cols_2) > 1 else ""
        amount = cols_2[2].text.strip() if len(cols_2) > 2 else ""

        cols_1 = row.find_elements(By.CSS_SELECTOR, ".col-md-1")
        status = cols_1[0].text.strip() if len(cols_1) > 0 else ""
        status_date = cols_1[1].text.strip() if len(cols_1) > 1 else ""

        all_projects.append({
            "Project Name": name,
            "Country": country,
            "Sector": sector,
            "Signed Amount": amount,
            "Status": status,
            "Status Date": status_date
        })

    time.sleep(1)  # Polite delay

driver.quit()

# Save to CSV
df = pd.DataFrame(all_projects)
df.to_csv("eib_all_projects.csv", index=False)
print("Done! All data saved to eib_all_projects.csv")


Scraping page 1 of 671...
Scraping page 2 of 671...
Scraping page 3 of 671...
Scraping page 4 of 671...
Scraping page 5 of 671...
Scraping page 6 of 671...
Scraping page 7 of 671...
Scraping page 8 of 671...
Scraping page 9 of 671...
Scraping page 10 of 671...
Scraping page 11 of 671...
Scraping page 12 of 671...
Scraping page 13 of 671...
Scraping page 14 of 671...
Scraping page 15 of 671...
Scraping page 16 of 671...
Scraping page 17 of 671...
Scraping page 18 of 671...
Scraping page 19 of 671...
Scraping page 20 of 671...
Scraping page 21 of 671...
Scraping page 22 of 671...
Scraping page 23 of 671...
Scraping page 24 of 671...
Scraping page 25 of 671...
Scraping page 26 of 671...
Scraping page 27 of 671...
Scraping page 28 of 671...
Scraping page 29 of 671...
Scraping page 30 of 671...
Scraping page 31 of 671...
Scraping page 32 of 671...
Scraping page 33 of 671...
Scraping page 34 of 671...
Scraping page 35 of 671...
Scraping page 36 of 671...
Scraping page 37 of 671...
Scraping p

In [6]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

BASE_URL = "https://www.eib.org/en/projects/all/index"
TOTAL_PAGES = 5  # 0-based: 0 to 670
ITEMS_PER_PAGE = 25

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

all_projects = []

for page_num in range(TOTAL_PAGES):
    print(f"Scraping page {page_num+1} of {TOTAL_PAGES}...")
    url = (
        f"{BASE_URL}?q=&sortColumn=null&sortDir=asc&pageNumber={page_num}"
        f"&itemPerPage={ITEMS_PER_PAGE}&pageable=true&la=EN&deLa=EN"
        "&yearFrom=&orYearFrom=true&yearTo=&orYearTo=true&orStatus=true"
        "&orRegions=true&orCountries=true&orSectors=true"
    )
    driver.get(url)
    time.sleep(2)  # Wait for JS to load

    rows = driver.find_elements(By.CSS_SELECTOR, "div.row-list.row-list-standard.row-contained")
    for row in rows:
        try:
            # Project Name
            name = row.find_element(By.CSS_SELECTOR, ".col-md-4 a").text.strip()
        except:
            continue

        # Country, Sector, Amount
        cols_2 = row.find_elements(By.CSS_SELECTOR, ".col-md-2")
        country = cols_2[0].text.strip() if len(cols_2) > 0 else ""
        sector = cols_2[1].text.strip() if len(cols_2) > 1 else ""
        amount = cols_2[2].text.strip() if len(cols_2) > 2 else ""

        # Clean up amount: fix euro symbol and remove extra spaces/commas
        amount = (
            amount.replace('\xa0', ' ')
            .replace('â‚¬', '€')
            .replace('EUR', '€')
            .replace(',', '')
            .strip()
        )

        # Status, Status Date
        cols_1 = row.find_elements(By.CSS_SELECTOR, ".col-md-1")
        status = cols_1[0].text.strip() if len(cols_1) > 0 else ""
        status_date = cols_1[1].text.strip() if len(cols_1) > 1 else ""

        # Add to list, ensuring all fields are strings
        all_projects.append({
            "Project Name": name,
            "Country": country,
            "Sector": sector,
            "Signed Amount": amount,
            "Status": status,
            "Status Date": status_date
        })

    time.sleep(1)  # Polite delay

driver.quit()

# Save to CSV with proper quoting and UTF-8 encoding
df = pd.DataFrame(all_projects)
df.to_csv("eib_all_projects_clean.csv", index=False, encoding="utf-8", quoting=1)
print("Done! All data saved to eib_all_projects_clean.csv")


Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


Scraping page 1 of 5...
Scraping page 2 of 5...
Scraping page 3 of 5...
Scraping page 4 of 5...
Scraping page 5 of 5...
Done! All data saved to eib_all_projects_clean.csv


In [7]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

BASE_URL = "https://www.eib.org/en/projects/all/index"
TOTAL_PAGES = 5  # Try with 5 pages first
ITEMS_PER_PAGE = 25

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

with open("eib_projects_vertical.txt", "w", encoding="utf-8") as f:
    # Write header block
    f.write("Title\nCountries\nSectors\nSigned Amount\nCurrent status\nLast status change\n")

    for page_num in range(TOTAL_PAGES):
        print(f"Scraping page {page_num+1} of {TOTAL_PAGES}...")
        url = (
            f"{BASE_URL}?q=&sortColumn=null&sortDir=asc&pageNumber={page_num}"
            f"&itemPerPage={ITEMS_PER_PAGE}&pageable=true&la=EN&deLa=EN"
            "&yearFrom=&orYearFrom=true&yearTo=&orYearTo=true&orStatus=true"
            "&orRegions=true&orCountries=true&orSectors=true"
        )
        driver.get(url)
        time.sleep(2)  # Wait for JS to load

        rows = driver.find_elements(By.CSS_SELECTOR, "div.row-list.row-list-standard.row-contained")
        for row in rows:
            try:
                name = row.find_element(By.CSS_SELECTOR, ".col-md-4 a").text.strip()
            except:
                continue

            cols_2 = row.find_elements(By.CSS_SELECTOR, ".col-md-2")
            country = cols_2[0].text.strip() if len(cols_2) > 0 else ""
            sector = cols_2[1].text.strip() if len(cols_2) > 1 else ""
            amount = cols_2[2].text.strip() if len(cols_2) > 2 else ""
            amount = (
                amount.replace('\xa0', ' ')
                .replace('â‚¬', '€')
                .replace('EUR', '€')
                .replace(',', '')
                .strip()
            )

            cols_1 = row.find_elements(By.CSS_SELECTOR, ".col-md-1")
            status = cols_1[0].text.strip() if len(cols_1) > 0 else ""
            status_date = cols_1[1].text.strip() if len(cols_1) > 1 else ""

            # Write each field on its own line
            f.write(f"{name}\n{country}\n{sector}\n{amount}\n{status}\n{status_date}\n\n")  # Blank line between projects

        time.sleep(1)  # Polite delay

driver.quit()
print("Done! All data saved to eib_projects_vertical.txt")


Scraping page 1 of 5...
Scraping page 2 of 5...
Scraping page 3 of 5...
Scraping page 4 of 5...
Scraping page 5 of 5...
Done! All data saved to eib_projects_vertical.txt


In [9]:
print("\nDone! Data collected and checked. Vertical text file saved to eib_projects_vertical.txt")


Done! Data collected and checked. Vertical text file saved to eib_projects_vertical.txt


In [11]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# Setup headless Chrome
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

base_url = "https://www.eib.org/en/projects/all/index?q=&sortColumn=null&sortDir=asc&pageNumber={}&itemPerPage=25&pageable=true&la=EN&deLa=EN"
all_data = []

page_number = 0
max_pages = 1000  # adjust as needed

while True:
    print(f"Scraping page {page_number + 1}...")
    driver.get(base_url.format(page_number))
    time.sleep(3)  # Let JS load
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    articles = soup.select("div#mainlist article")

    if not articles:
        print("No more articles found. Ending scrape.")
        break

    for article in articles:
        try:
            title = article.select_one("h3.row-title a").text.strip()
            link = "https://www.eib.org" + article.select_one("h3.row-title a")["href"]
            country = article.select("div.row-tags")[0].text.strip()
            sector = article.select("div.row-tags")[1].text.strip()
            amount = article.select("div.col-md-2.col-xs-6")[2].text.strip()
            status = article.select("div.col-md-1.col-xs-6")[0].text.strip()
            status_date = article.select("div.col-md-1.col-xs-6")[1].text.strip()
        except Exception as e:
            print(f"Skipping a row due to error: {e}")
            continue
        
        all_data.append({
            "Title": title,
            "Link": link,
            "Country": country,
            "Sector": sector,
            "Signed Amount": amount,
            "Status": status,
            "Status Date": status_date
        })

    page_number += 1
    if page_number >= max_pages:
        print("Reached max page limit.")
        break

driver.quit()

# Export to Excel
df = pd.DataFrame(all_data)
df.to_excel("eib_projects2.xlsx", index=False)
print("Scraping completed and saved to eib_projects2.xlsx")


Scraping page 1...
No more articles found. Ending scrape.
Scraping completed and saved to eib_projects2.xlsx


In [12]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# Setup headless browser
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

all_data = []
page_number = 0
total_results = 16769
items_per_page = 25
total_pages = (total_results // items_per_page) + 1

while page_number < total_pages:
    print(f"Scraping page {page_number + 1} of {total_pages}...")
    url = f"https://www.eib.org/en/projects/all/index?q=&sortColumn=null&sortDir=asc&pageNumber={page_number}&itemPerPage=25&pageable=true&la=EN&deLa=EN"
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    articles = soup.select("div#mainlist article")

    if not articles:
        print("No more articles found. Stopping early.")
        break

    for article in articles:
        try:
            title = article.select_one("h3.row-title a").text.strip()
            link = "https://www.eib.org" + article.select_one("h3.row-title a")["href"]
            country = article.select("div.row-tags")[0].text.strip()
            sector = article.select("div.row-tags")[1].text.strip()
            amount = article.select("div.col-md-2.col-xs-6")[2].text.strip()
            status = article.select("div.col-md-1.col-xs-6")[0].text.strip()
            status_date = article.select("div.col-md-1.col-xs-6")[1].text.strip()
        except Exception as e:
            print(f"Skipping project due to error: {e}")
            continue

        all_data.append({
            "Title": title,
            "Link": link,
            "Country": country,
            "Sector": sector,
            "Signed Amount": amount,
            "Status": status,
            "Status Date": status_date
        })

    page_number += 1

driver.quit()

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel("eib_all_projects.xlsx", index=False)
print(f"✅ Scraping completed. {len(df)} records saved to 'eib_all_projects.xlsx'")


Scraping page 1 of 671...
No more articles found. Stopping early.
✅ Scraping completed. 0 records saved to 'eib_all_projects.xlsx'


In [13]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup Selenium WebDriver
options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

all_data = []
page_number = 0
total_pages = 671  # based on your inspection

while page_number < total_pages:
    print(f"Scraping page {page_number + 1} of {total_pages}...")

    url = f"https://www.eib.org/en/projects/all/index?q=&sortColumn=null&sortDir=asc&pageNumber={page_number}&itemPerPage=25&pageable=true&la=EN&deLa=EN"
    driver.get(url)

    try:
        # Wait until the project list is present
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "mainlist"))
        )
    except Exception as e:
        print(f"Timeout on page {page_number + 1}. Error: {e}")
        break

    soup = BeautifulSoup(driver.page_source, "html.parser")
    articles = soup.select("div#mainlist article")

    if not articles:
        print("No articles found — stopping.")
        break

    for article in articles:
        try:
            title = article.select_one("h3.row-title a").text.strip()
            link = "https://www.eib.org" + article.select_one("h3.row-title a")["href"]
            country = article.select("div.row-tags")[0].text.strip()
            sector = article.select("div.row-tags")[1].text.strip()
            amount = article.select("div.col-md-2.col-xs-6")[2].text.strip()
            status = article.select("div.col-md-1.col-xs-6")[0].text.strip()
            status_date = article.select("div.col-md-1.col-xs-6")[1].text.strip()
        except Exception as e:
            print(f"Error parsing a project: {e}")
            continue

        all_data.append({
            "Title": title,
            "Link": link,
            "Country": country,
            "Sector": sector,
            "Signed Amount": amount,
            "Status": status,
            "Status Date": status_date
        })

    page_number += 1

driver.quit()

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel("eib_all_projects.xlsx", index=False)
print(f"✅ Finished scraping {len(df)} records. Saved to 'eib_all_projects.xlsx'")


Scraping page 1 of 671...
No articles found — stopping.
✅ Finished scraping 0 records. Saved to 'eib_all_projects.xlsx'


In [2]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup headless Chrome
options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

# Prepare list to store results
all_data = []

# Set number of pages to scrape (each page has 25 items)
total_pages = 671  # Change to 671 to scrape all
base_url = "https://www.eib.org/en/projects/all/index?pageNumber={}&itemPerPage=25"

for page in range(total_pages):
    print(f"Scraping page {page + 1} of {total_pages}...")

    driver.get(base_url.format(page))
    
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div#mainlist h3.row-title a"))
        )
    except:
        print("❌ Timeout or page failed to load.")
        break

    soup = BeautifulSoup(driver.page_source, "html.parser")
    rows = soup.select("div.row-list.row-list-standard.row-contained")

    for row in rows:
        try:
            title_tag = row.select_one("h3.row-title a")
            title = title_tag.text.strip()
            link = "https://www.eib.org" + title_tag.get("href", "")

            columns = row.find_all("div", class_="col-md-2") + row.find_all("div", class_="col-md-1")

            country = columns[0].get_text(strip=True)
            sector = columns[1].get_text(strip=True)
            amount = columns[2].get_text(strip=True)
            status = columns[3].get_text(strip=True)
            status_date = columns[4].get_text(strip=True)

            all_data.append({
                "Title": title,
                "Link": link,
                "Country": country,
                "Sector": sector,
                "Signed Amount": amount,
                "Status": status,
                "Status Date": status_date
            })

        except Exception as e:
            print(f"⚠️ Error parsing row: {e}")
            continue

driver.quit()

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel("eib_projects_eib.xlsx", index=False)
print(f"✅ Finished scraping {len(df)} projects. Saved to 'eib_projects_eib.xlsx'")


Scraping page 1 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 2 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 3 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 4 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 5 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 6 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 7 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 8 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 9 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 10 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping page 11 of 671...
⚠️ Error parsing row: 'NoneType' object has no attribute 'text'
Scraping

In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup headless Chrome
options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

# Prepare storage
all_data = []
skipped_rows = 0

# Set number of pages to scrape
total_pages = 671  # Change to 671 for all pages
base_url = "https://www.eib.org/en/projects/all/index?pageNumber={}&itemPerPage=25"

for page in range(total_pages):
    print(f"\n🔄 Scraping page {page + 1} of {total_pages}...")

    driver.get(base_url.format(page))

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div#mainlist h3.row-title a"))
        )
    except:
        print("❌ Timeout or page failed to load.")
        continue

    soup = BeautifulSoup(driver.page_source, "html.parser")
    rows = soup.select("div.row-list.row-list-standard.row-contained")

    for row in rows:
        try:
            # Get title and link
            title_tag = row.select_one("h3.row-title a")
            if title_tag:
                title = title_tag.get_text(strip=True)
                link = "https://www.eib.org" + title_tag.get("href", "")
            else:
                title = ""
                link = ""
                skipped_rows += 1
                print("⚠️ Skipped row due to missing title.")
                continue  # Title is mandatory to include the row

            # Collect other columns (may vary)
            cols = row.find_all("div", class_="col-md-2") + row.find_all("div", class_="col-md-1")

            def safe_get(index):
                try:
                    return cols[index].get_text(strip=True)
                except:
                    return ""

            country = safe_get(0)
            sector = safe_get(1)
            amount = safe_get(2)
            status = safe_get(3)
            status_date = safe_get(4)

            all_data.append({
                "Title": title,
                "Link": link,
                "Country": country,
                "Sector": sector,
                "Signed Amount": amount,
                "Status": status,
                "Status Date": status_date
            })

        except Exception as e:
            skipped_rows += 1
            print(f"⚠️ Error parsing row: {e}")
            continue

    time.sleep(1)  # Be polite

driver.quit()

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel("eib_projects.xlsx", index=False)
print(f"\n✅ Finished scraping {len(df)} projects. Skipped rows: {skipped_rows}")
print("📁 Saved to 'eib_projects_final.xlsx'")



🔄 Scraping page 1 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 2 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 3 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 4 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 5 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 6 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 7 of 671...
⚠️ Skipped row due to missing title.

🔄 Scraping page 8 of 671...


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup headless Chrome
options = Options()
options.add_argument("--headless=new")  # Important for newer Chrome versions
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

all_data = []
skipped_rows = 0
total_pages = 671  # Set to 671 to scrape all pages
base_url = "https://www.eib.org/en/projects/all/index?pageNumber={}&itemPerPage=25"

for page in range(total_pages):
    print(f"\n🔄 Scraping page {page + 1} of {total_pages}...")

    driver.get(base_url.format(page))

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.row-list-standard"))
        )
        time.sleep(2)
        rows = driver.find_elements(By.CSS_SELECTOR, "div.row-list-standard")
    except Exception as e:
        print(f"❌ Timeout or load error: {e}")
        continue

    for row in rows:
        try:
            # Get all child divs inside the row
            all_divs = row.find_elements(By.CSS_SELECTOR, "div")

            # Extract title + link from the <a> tag inside the first <div>
            try:
                title_element = row.find_element(By.CSS_SELECTOR, "h3.row-title a")
                title = title_element.text.strip()
                link = "https://www.eib.org" + title_element.get_attribute("href")
            except:
                title = ""
                link = ""
                skipped_rows += 1
                print("⚠️ Skipped row: no title or link found.")
                continue

            # Extract 5 expected data columns
            country = all_divs[1].text.strip() if len(all_divs) > 1 else ""
            sector = all_divs[2].text.strip() if len(all_divs) > 2 else ""
            amount = all_divs[3].text.strip() if len(all_divs) > 3 else ""
            status = all_divs[4].text.strip() if len(all_divs) > 4 else ""
            status_date = all_divs[5].text.strip() if len(all_divs) > 5 else ""

            all_data.append({
                "Title": title,
                "Link": link,
                "Country": country,
                "Sector": sector,
                "Signed Amount": amount,
                "Status": status,
                "Status Date": status_date
            })

        except Exception as e:
            skipped_rows += 1
            print(f"⚠️ Error parsing row: {e}")
            continue

driver.quit()

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel("eib_projects8.xlsx", index=False)
print(f"\n✅ Scraped {len(df)} projects. Skipped rows: {skipped_rows}")
print("📁 Saved to 'eib_projects8.xlsx'")



🔄 Scraping page 1 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 2 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 3 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 4 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 5 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 6 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 7 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 8 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 9 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 10 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 11 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 12 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 13 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 14 of 671...
⚠️ Skipped row: no title or link found.

🔄 Scraping page 15 of 671...